# Biermann (1951): Comet Tails and Solar Corpuscular Radiation
# Biermann (1951): 혜성 꼬리와 태양 입자 복사

## Implementation of Key Concepts / 핵심 개념 구현

This notebook implements and visualizes the key physical arguments from Biermann's 1951 paper,
which first proposed that comet ion tails are driven by continuous solar corpuscular radiation
(now called the solar wind), not by radiation pressure.

이 노트북은 Biermann의 1951년 논문의 핵심 물리적 논증을 구현하고 시각화합니다.
이 논문은 혜성 이온 꼬리가 광압이 아닌 연속적인 태양 입자 복사(현재 태양풍)에 의해
구동된다고 최초로 제안했습니다.

**Contents / 목차:**
1. Radiation Pressure vs Observed Repulsive Force / 광압 vs 관측된 반발력
2. Three-Component Plasma Momentum Transfer / 3-성분 플라즈마 운동량 전달
3. Electron vs Proton Momentum Transfer / 전자 vs 양성자 운동량 전달 비교
4. CO⁺ Ion Acceleration by Solar Corpuscular Radiation / 태양 입자 복사에 의한 CO⁺ 이온 가속
5. Comet Tail Geometry and Aberration / 혜성 꼬리 기하학과 수차
6. Solar Corpuscular Radiation Density Profile / 태양 입자 복사 밀도 분포
7. Connection to Modern Solar Wind Measurements / 현대 태양풍 관측과의 비교

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, Circle
import matplotlib.patches as mpatches

plt.rcParams.update({
    'figure.figsize': (10, 6),
    'font.size': 12,
    'axes.labelsize': 13,
    'axes.titlesize': 14,
    'legend.fontsize': 11,
    'figure.dpi': 100,
})

# Physical constants in CGS
e_cgs = 4.803e-10       # Elementary charge [esu]
m_e = 9.109e-28         # Electron mass [g]
m_p = 1.673e-24         # Proton mass [g]
m_CO = 28 * m_p         # CO+ molecular mass [g] (~28 amu)
k_B = 1.381e-16         # Boltzmann constant [erg/K]
c_light = 2.998e10      # Speed of light [cm/s]
G_grav = 6.674e-8       # Gravitational constant [cm³/g/s²]
M_sun = 1.989e33        # Solar mass [g]
R_sun = 6.957e10        # Solar radius [cm]
AU = 1.496e13           # 1 AU [cm]

print("Constants loaded successfully / 상수 로드 완료")

## Part 1: Radiation Pressure vs Observed Repulsive Force
## 파트 1: 광압 vs 관측된 반발력

Biermann's starting point: the repulsive force factor $\mu$ (ratio of repulsive force to gravity)
observed in Type I comet tails is $\mu \approx 18$, but radiation pressure can only provide
$\mu = 47f$ where $f$ is the oscillator strength of CO⁺ transitions ($f \sim 10^{-2}$ to $10^{-1}$).

Biermann의 출발점: Type I 혜성 꼬리에서 관측된 반발력 인자 $\mu$(반발력/중력)는
$\mu \approx 18$이지만, 광압은 $\mu = 47f$만 제공할 수 있고 CO⁺의 oscillator strength
$f \sim 10^{-2}$–$10^{-1}$이므로 크게 부족합니다.

In [ ]:
# Bredichin classification: observed mu values for each tail type
# Bredichin 분류: 각 꼬리 유형의 관측된 mu 값
tail_types = ['Type I\n(Ion tail / 이온 꼬리)', 'Type II\n(Dust tail / 먼지 꼬리)', 'Type III\n(Short tail / 짧은 꼬리)']
mu_observed = [18, 1.25, 0.15]  # Bredichin typical values

# Radiation pressure prediction: mu = 47 * f
# Oscillator strengths for relevant CO+ transitions
f_values = np.logspace(-2, -0.5, 50)
mu_radiation = 47 * f_values

# Specific CO+ oscillator strength estimates
f_CO_low = 0.01
f_CO_high = 0.1
mu_rad_low = 47 * f_CO_low   # = 0.47
mu_rad_high = 47 * f_CO_high  # = 4.7

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left panel: Bar chart of observed mu values
ax1 = axes[0]
colors = ['#e74c3c', '#f39c12', '#3498db']
bars = ax1.bar(tail_types, mu_observed, color=colors, edgecolor='black', linewidth=1.2, alpha=0.8)
ax1.axhspan(mu_rad_low, mu_rad_high, alpha=0.3, color='green',
            label=f'Radiation pressure range\n(μ = 47f, f = {f_CO_low}–{f_CO_high})')
ax1.set_ylabel('Repulsive force factor μ\n(반발력 인자 μ)')
ax1.set_title('Observed μ vs Radiation Pressure Prediction\n관측된 μ vs 광압 예측')
ax1.legend(loc='upper right')
ax1.set_ylim(0, 22)
for bar, val in zip(bars, mu_observed):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'μ = {val}', ha='center', fontweight='bold')

# Right panel: The discrepancy for Type I
ax2 = axes[1]
ax2.semilogy(f_values, mu_radiation, 'g-', linewidth=2, label='μ = 47f (radiation pressure / 광압)')
ax2.axhline(y=18, color='red', linestyle='--', linewidth=2, label='Observed Type I: μ = 18 (관측값)')
ax2.axvspan(f_CO_low, f_CO_high, alpha=0.2, color='blue', label='Realistic f range for CO⁺')
ax2.fill_between(f_values, mu_radiation, 18,
                 where=(mu_radiation < 18), alpha=0.15, color='red')
ax2.set_xlabel('Oscillator strength f')
ax2.set_ylabel('Repulsive force factor μ')
ax2.set_title('The Discrepancy: Radiation Pressure Cannot\nExplain Type I Tails / 광압의 실패')
ax2.legend(loc='lower right', fontsize=10)
ax2.set_xlim(0.01, 0.3)
ax2.set_ylim(0.3, 30)
ax2.annotate(f'Gap: ×{18/mu_rad_high:.0f}–×{18/mu_rad_low:.0f}',
             xy=(0.03, 5), fontsize=14, fontweight='bold', color='red',
             bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.8))

plt.tight_layout()
plt.savefig('part1_radiation_pressure_discrepancy.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nRadiation pressure prediction / 광압 예측:")
print(f"  f = {f_CO_low}: μ = {mu_rad_low:.2f}")
print(f"  f = {f_CO_high}: μ = {mu_rad_high:.1f}")
print(f"  Observed Type I / 관측 Type I: μ = {mu_observed[0]}")
print(f"  Discrepancy / 불일치: ×{18/mu_rad_high:.0f} to ×{18/mu_rad_low:.0f}")

## Part 2: Three-Component Plasma Momentum Transfer
## 파트 2: 3-성분 플라즈마 운동량 전달

Biermann modeled the interaction as a three-component plasma: cometary molecules ($m$),
solar protons ($p$), and electrons ($e$). The momentum equations (Eq. 1) describe how
collisions between these species transfer momentum.

Biermann은 상호작용을 3-성분 플라즈마로 모델링했습니다: 혜성 분자($m$), 태양 양성자($p$),
전자($e$). 운동 방정식(Eq. 1)은 이 종들 간의 충돌이 운동량을 전달하는 방식을 기술합니다.

Key insight: The collision frequency $\gamma$ is related to the electrical conductivity $\sigma$ via:

$$\gamma_{em} + \gamma_{ep} = \frac{e^2 n_e}{\sigma m_e}$$

핵심 통찰: 충돌 빈도 $\gamma$는 전기 전도도 $\sigma$와 다음과 같이 연결됩니다.

In [ ]:
def coulomb_cross_section(charge_1, charge_2, reduced_mass, v_rel, ln_lambda=10):
    """Compute Coulomb collision cross-section.

    Args:
        charge_1: Charge of particle 1 [esu].
        charge_2: Charge of particle 2 [esu].
        reduced_mass: Reduced mass [g].
        v_rel: Relative velocity [cm/s].
        ln_lambda: Coulomb logarithm (default ~10 for solar wind plasma).

    Returns:
        Cross-section [cm²].
    """
    # Impact parameter for 90-degree deflection
    b_90 = charge_1 * charge_2 / (reduced_mass * v_rel**2)
    # Coulomb cross-section with logarithmic correction
    sigma = np.pi * b_90**2 * ln_lambda
    return sigma


def gas_kinetic_cross_section(radius_cm=1.5e-8):
    """Compute gas-kinetic (hard sphere) collision cross-section.

    Args:
        radius_cm: Effective molecular radius [cm].

    Returns:
        Cross-section [cm²].
    """
    return np.pi * (2 * radius_cm)**2


def spitzer_conductivity(T, n_e, Z=1, ln_lambda=10):
    """Compute Spitzer electrical conductivity of a plasma.

    Args:
        T: Temperature [K].
        n_e: Electron density [cm⁻³].
        Z: Ion charge number.
        ln_lambda: Coulomb logarithm.

    Returns:
        Conductivity [esu] (Gaussian units).
    """
    # Spitzer formula: sigma = (kT)^(3/2) / (pi * Z * e^2 * sqrt(m_e) * ln_lambda)
    # Simplified numerical coefficient
    return 1.53e-2 * T**1.5 / (Z * ln_lambda)  # in s⁻¹ (CGS)


# Compare Coulomb vs gas-kinetic cross-sections
v_rel = np.logspace(6, 9, 100)  # Relative velocity [cm/s]

# Electron-CO+ Coulomb cross-section
mu_e_CO = m_e * m_CO / (m_e + m_CO)  # ≈ m_e
sigma_coulomb_e = coulomb_cross_section(e_cgs, e_cgs, mu_e_CO, v_rel)

# Proton-CO+ Coulomb cross-section
mu_p_CO = m_p * m_CO / (m_p + m_CO)
sigma_coulomb_p = coulomb_cross_section(e_cgs, e_cgs, mu_p_CO, v_rel)

# Gas-kinetic cross-section (constant)
sigma_gk = gas_kinetic_cross_section()

fig, ax = plt.subplots(figsize=(10, 6))

ax.loglog(v_rel / 1e5, sigma_coulomb_e, 'r-', linewidth=2.5,
          label='Coulomb: e⁻ ↔ CO⁺ (전자-분자)')
ax.loglog(v_rel / 1e5, sigma_coulomb_p, 'b-', linewidth=2.5,
          label='Coulomb: p⁺ ↔ CO⁺ (양성자-분자)')
ax.axhline(y=sigma_gk, color='green', linestyle='--', linewidth=2,
           label=f'Gas-kinetic / 가스 운동학: σ = {sigma_gk:.1e} cm²')

# Mark typical solar wind velocity
v_sw_typ = 500  # km/s
ax.axvline(x=v_sw_typ, color='gray', linestyle=':', linewidth=1.5,
           label=f'Typical solar wind: {v_sw_typ} km/s')

# Mark the regime Biermann analyzed
ax.axvspan(300, 2000, alpha=0.1, color='orange', label='Biermann regime\n(500–2000 km/s)')

ax.set_xlabel('Relative velocity [km/s] / 상대 속도')
ax.set_ylabel('Collision cross-section [cm²] / 충돌 단면적')
ax.set_title('Coulomb vs Gas-Kinetic Cross-Sections\nCoulomb vs 가스 운동학적 충돌 단면적')
ax.legend(loc='upper right', fontsize=10)
ax.set_xlim(10, 1e4)
ax.set_ylim(1e-22, 1e-10)
ax.grid(True, alpha=0.3)

# Annotate the key point
idx_500 = np.argmin(np.abs(v_rel/1e5 - 500))
ratio_e = sigma_coulomb_e[idx_500] / sigma_gk
ax.annotate(f'At 500 km/s:\nσ_Coulomb(e⁻)/σ_GK ≈ {ratio_e:.0e}',
            xy=(500, sigma_coulomb_e[idx_500]),
            xytext=(50, 1e-13),
            arrowprops=dict(arrowstyle='->', color='red'),
            fontsize=11, color='red', fontweight='bold')

plt.tight_layout()
plt.savefig('part2_cross_sections.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nAt v = 500 km/s:")
print(f"  Coulomb (e⁻-CO⁺): σ = {sigma_coulomb_e[idx_500]:.2e} cm²")
print(f"  Coulomb (p⁺-CO⁺): σ = {sigma_coulomb_p[idx_500]:.2e} cm²")
print(f"  Gas-kinetic:       σ = {sigma_gk:.2e} cm²")
print(f"  Ratio e⁻ Coulomb / GK: {ratio_e:.1e}")
print(f"\n→ Electron Coulomb cross-section is ENORMOUSLY larger than gas-kinetic!")
print(f"→ 전자 Coulomb 단면적이 가스 운동학적 단면적보다 압도적으로 큽니다!")

## Part 3: Electron vs Proton Momentum Transfer Efficiency
## 파트 3: 전자 vs 양성자 운동량 전달 효율

Biermann's key finding: despite being much lighter, **electrons** are far more efficient
at transferring momentum to cometary ions than protons. This is because:

1. Electron Coulomb cross-sections scale as $\sigma \propto 1/(\mu_{red}^2 v^4)$, and $\mu_{red} \approx m_e$ for e⁻–CO⁺
2. The momentum transferred per collision is small (low $m_e$), but the collision rate is enormous

Biermann의 핵심 발견: 훨씬 가벼움에도 불구하고, **전자**가 양성자보다 혜성 이온에
운동량을 훨씬 효율적으로 전달합니다. 이는:

1. 전자 Coulomb 단면적이 $\sigma \propto 1/(\mu_{red}^2 v^4)$로 스케일링되고, e⁻–CO⁺에서 $\mu_{red} \approx m_e$
2. 충돌당 전달되는 운동량은 작지만($m_e$ 작음), 충돌 빈도가 압도적으로 높음

In [ ]:
def momentum_transfer_rate(m_projectile, m_target, n_projectile, v_projectile, sigma_func):
    """Compute momentum transfer rate (force per target particle).

    The rate of momentum transfer = n * v * sigma * delta_p_per_collision
    For Coulomb scattering, delta_p ~ 2 * mu_red * v (for 90-degree deflection).

    Args:
        m_projectile: Mass of projectile particle [g].
        m_target: Mass of target particle [g].
        n_projectile: Number density of projectiles [cm⁻³].
        v_projectile: Velocity of projectile [cm/s].
        sigma_func: Function returning cross-section given velocity.

    Returns:
        Acceleration of target [cm/s²].
    """
    mu_red = m_projectile * m_target / (m_projectile + m_target)
    sigma = sigma_func(v_projectile)
    # Force per target = n * sigma * v * (2 * mu_red * v) / m_target
    # Simplified: dv/dt = n * sigma * v * 2 * mu_red * v / m_target
    force_per_target = n_projectile * sigma * v_projectile * 2 * mu_red * v_projectile
    acceleration = force_per_target / m_target
    return acceleration


# Biermann's parameters (Eq. 6 in the paper)
n_p = 1e4        # Solar proton density [cm⁻³] (log n_p = 4)
n_e = n_p        # Quasi-neutrality
v_p = 1e8        # Solar proton velocity [cm/s] (1000 km/s, log v_p = 8)
n_m = 10         # Cometary molecule density [cm⁻³] (log n_m = 1)
T = 1e4          # Temperature [K]
sigma_cond = 10**12.7  # Electrical conductivity [esu]

# Method 1: Direct Coulomb collision calculation
# Electron contribution: using Biermann's formula dv_m/dt = (e²/σm_m) * n_e * v_e
# At T = 10^4 K, electron thermal velocity
v_e_thermal = np.sqrt(k_B * T / m_e)

# Biermann's Eq. 6: dv_m/dt via electrical conductivity
accel_biermann = (e_cgs**2 / (sigma_cond * m_CO)) * n_e * v_p
log_accel_biermann = np.log10(accel_biermann)

# Method 2: Direct Coulomb collision approach for comparison
# Electron-CO+ via Coulomb
sigma_e_CO_func = lambda v: coulomb_cross_section(e_cgs, e_cgs, m_e, v)
accel_electron = momentum_transfer_rate(m_e, m_CO, n_e, v_p, sigma_e_CO_func)

# Proton-CO+ via gas-kinetic only (as Biermann argues)
sigma_p_CO_gk = lambda v: gas_kinetic_cross_section()
accel_proton_gk = momentum_transfer_rate(m_p, m_CO, n_p, v_p, sigma_p_CO_gk)

# Proton-CO+ via Coulomb (for comparison)
sigma_p_CO_coul = lambda v: coulomb_cross_section(e_cgs, e_cgs, m_p * m_CO / (m_p + m_CO), v)
accel_proton_coul = momentum_transfer_rate(m_p, m_CO, n_p, v_p, sigma_p_CO_coul)

print("=" * 65)
print("Momentum Transfer Comparison / 운동량 전달 비교")
print("=" * 65)
print(f"\nBiermann's parameters / Biermann의 매개변수:")
print(f"  n_p = {n_p:.0e} cm⁻³, v_p = {v_p:.0e} cm/s ({v_p/1e5:.0f} km/s)")
print(f"  n_m = {n_m:.0f} cm⁻³, T = {T:.0e} K")
print(f"  σ (conductivity) = {sigma_cond:.2e} esu")
print(f"\nCO⁺ acceleration / CO⁺ 가속도:")
print(f"  Biermann's formula (Eq. 6): dv_m/dt = {accel_biermann:.2e} cm/s²")
print(f"    → log₁₀(dv_m/dt) = {log_accel_biermann:.1f}")
print(f"    → Biermann's result: 10^(3.0 ± 1.5) cm/s²")
print(f"\n  Electron Coulomb:     {accel_electron:.2e} cm/s²")
print(f"  Proton gas-kinetic:   {accel_proton_gk:.2e} cm/s²")
print(f"  Proton Coulomb:       {accel_proton_coul:.2e} cm/s²")
print(f"\n  Ratio electron/proton(GK): {accel_electron/accel_proton_gk:.0e}")
print(f"  → Electrons dominate by orders of magnitude! / 전자가 수 자릿수 지배!")

# Visualization: bar chart comparing mechanisms
fig, ax = plt.subplots(figsize=(10, 6))
mechanisms = ['Electron\nCoulomb\n(전자 Coulomb)',
              'Proton\nCoulomb\n(양성자 Coulomb)',
              'Proton\nGas-kinetic\n(양성자 가스운동학)',
              "Biermann's\nformula\n(Biermann 공식)"]
accels = [accel_electron, accel_proton_coul, accel_proton_gk, accel_biermann]
colors_bar = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6']

bars = ax.bar(mechanisms, accels, color=colors_bar, edgecolor='black', linewidth=1.2, alpha=0.8)
ax.set_yscale('log')
ax.set_ylabel('CO⁺ acceleration [cm/s²]\nCO⁺ 가속도')
ax.set_title("Momentum Transfer Mechanisms Compared\n운동량 전달 메커니즘 비교")
ax.axhline(y=1e4, color='red', linestyle='--', linewidth=2, alpha=0.7,
           label='Observed: ~10⁴ cm/s² (관측값)')
ax.legend(fontsize=12)
ax.set_ylim(1e-5, 1e8)
ax.grid(True, alpha=0.3, axis='y')

for bar, val in zip(bars, accels):
    ax.text(bar.get_x() + bar.get_width()/2, val * 2,
            f'{val:.1e}', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('part3_momentum_transfer.png', dpi=150, bbox_inches='tight')
plt.show()

## Part 4: CO⁺ Ion Acceleration — Parameter Space Exploration
## 파트 4: CO⁺ 이온 가속 — 매개변수 공간 탐색

Biermann noted that his acceleration estimate has uncertainties of ±1.5 in log₁₀,
primarily from uncertainties in $n_p$ (±1) and $v_p$ (±0.5). Let's explore this
parameter space and compare with the observed acceleration.

Biermann은 가속도 추정값이 log₁₀에서 ±1.5의 불확실성을 가진다고 지적했습니다.
이는 주로 $n_p$(±1)와 $v_p$(±0.5)의 불확실성에서 비롯됩니다.
이 매개변수 공간을 탐색하고 관측 가속도와 비교합니다.

In [ ]:
# Parameter space exploration
# Biermann's Eq. 6: dv_m/dt = (e²/σm_m) * n_e * v_p
# log₁₀(dv_m/dt) = log₁₀(e²/σm_m) + log₁₀(n_e) + log₁₀(v_p)

log_n_range = np.linspace(2, 6, 100)   # log10(n_p) from 10² to 10⁶
log_v_range = np.linspace(7, 9, 100)   # log10(v_p) from 10⁷ to 10⁹

LOG_N, LOG_V = np.meshgrid(log_n_range, log_v_range)

# Biermann's formula
log_prefactor = np.log10(e_cgs**2 / (sigma_cond * m_CO))
LOG_ACCEL = log_prefactor + LOG_N + LOG_V

fig, ax = plt.subplots(figsize=(10, 7))

# Contour plot
levels = np.arange(-2, 10, 0.5)
cf = ax.contourf(LOG_N, LOG_V / 1, LOG_ACCEL, levels=levels, cmap='RdYlBu_r')
plt.colorbar(cf, ax=ax, label='log₁₀(dv_m/dt) [cm/s²]')

# Observed acceleration range
cs = ax.contour(LOG_N, LOG_V, LOG_ACCEL, levels=[3.5, 4.0, 4.5],
                colors=['white', 'yellow', 'white'], linewidths=2)
ax.clabel(cs, fmt='%.1f', fontsize=10)

# Biermann's nominal values
ax.plot(4, 8, 'k*', markersize=20, markeredgecolor='white', markeredgewidth=2,
        label="Biermann's estimate\n(log n_p=4, log v_p=8)")

# Biermann's uncertainty box
rect = plt.Rectangle((3, 7.5), 2, 1, linewidth=2, edgecolor='white',
                      facecolor='none', linestyle='--',
                      label='Biermann uncertainty\n(±1 in log n, ±0.5 in log v)')
ax.add_patch(rect)

# Modern solar wind values
ax.plot(np.log10(5), np.log10(4e7), 'g^', markersize=15, markeredgecolor='white',
        markeredgewidth=2, label='Modern slow wind\n(n~5, v~400 km/s)')
ax.plot(np.log10(3), np.log10(7e7), 'bs', markersize=12, markeredgecolor='white',
        markeredgewidth=2, label='Modern fast wind\n(n~3, v~700 km/s)')

ax.set_xlabel('log₁₀(n_p) [cm⁻³] — Proton density / 양성자 밀도')
ax.set_ylabel('log₁₀(v_p) [cm/s] — Proton velocity / 양성자 속도')
ax.set_title("CO⁺ Acceleration in Biermann's Parameter Space\nBiermann 매개변수 공간에서의 CO⁺ 가속도")
ax.legend(loc='lower right', fontsize=9)

# Add velocity scale on right axis
ax2 = ax.twinx()
ax2.set_ylim(ax.get_ylim())
v_ticks = [7.0, 7.5, 8.0, 8.5, 9.0]
ax2.set_yticks(v_ticks)
ax2.set_yticklabels([f'{10**(v)/1e5:.0f} km/s' for v in v_ticks])
ax2.set_ylabel('Velocity / 속도 [km/s]')

plt.tight_layout()
plt.savefig('part4_parameter_space.png', dpi=150, bbox_inches='tight')
plt.show()

print("Yellow contour = observed CO⁺ acceleration (~10⁴ cm/s²)")
print("노란 등고선 = 관측된 CO⁺ 가속도 (~10⁴ cm/s²)")
print("\nBiermann's estimate falls right in the observed range!")
print("Biermann의 추정값이 관측 범위 안에 정확히 들어갑니다!")

## Part 5: Comet Tail Geometry and Aberration Angle
## 파트 5: 혜성 꼬리 기하학과 수차각

Biermann derived the transverse velocity of comet tail molecules (Eq. in §5):

$$v_q = 42\sqrt{\frac{1\text{ AU}}{r}} \cos\varphi \quad [\text{km/s}]$$

The aberration angle between the tail and the radial direction depends on the ratio $v_q : v_p$.
In the corpuscular radiation theory, this angle is always small (a few degrees), consistent with
Hoffmeister's observations.

Biermann은 혜성 꼬리 분자의 횡단 속도를 유도했습니다 (§5의 수식).
꼬리와 태양 방향 사이의 수차각은 $v_q : v_p$ 비율에 따라 결정됩니다.

In [ ]:
def comet_transverse_velocity(r_au, phi_deg):
    """Compute transverse velocity of comet tail molecules (Biermann §5).

    For a parabolic orbit, the transverse velocity component is:
    v_q = 42 * sqrt(1 AU / r) * cos(phi) [km/s]

    Args:
        r_au: Heliocentric distance [AU].
        phi_deg: Angle between orbital tangent and Sun direction [degrees].

    Returns:
        Transverse velocity [km/s].
    """
    phi_rad = np.radians(phi_deg)
    return 42.0 * np.sqrt(1.0 / r_au) * np.cos(phi_rad)


def aberration_angle(v_q, v_sw):
    """Compute tail aberration angle.

    Args:
        v_q: Transverse velocity [km/s].
        v_sw: Solar wind radial velocity [km/s].

    Returns:
        Aberration angle [degrees].
    """
    return np.degrees(np.arctan(v_q / v_sw))


# Part A: Tail geometry visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: Transverse velocity vs heliocentric distance
ax1 = axes[0]
r_range = np.linspace(0.3, 5.0, 200)
for phi in [0, 30, 45, 60]:
    v_q = comet_transverse_velocity(r_range, phi)
    ax1.plot(r_range, v_q, linewidth=2, label=f'φ = {phi}°')

ax1.set_xlabel('Heliocentric distance r [AU]\n태양 중심 거리')
ax1.set_ylabel('Transverse velocity v_q [km/s]\n횡단 속도')
ax1.set_title('Comet Tail Transverse Velocity\n혜성 꼬리 횡단 속도')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0.3, 5)

# Right: Aberration angle for different solar wind speeds
ax2 = axes[1]
v_sw_values = [400, 500, 700, 1000, 2000]
colors_sw = plt.cm.viridis(np.linspace(0.2, 0.9, len(v_sw_values)))

for v_sw, color in zip(v_sw_values, colors_sw):
    v_q = comet_transverse_velocity(r_range, 0)  # phi=0 (maximum aberration)
    angle = aberration_angle(v_q, v_sw)
    ax2.plot(r_range, angle, linewidth=2, color=color,
             label=f'v_sw = {v_sw} km/s')

ax2.set_xlabel('Heliocentric distance r [AU]\n태양 중심 거리')
ax2.set_ylabel('Aberration angle [°]\n수차각')
ax2.set_title('Tail Aberration Angle (φ=0°)\n꼬리 수차각 (최대)')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)
ax2.set_xlim(0.3, 5)
ax2.set_ylim(0, 12)

plt.tight_layout()
plt.savefig('part5_tail_geometry.png', dpi=150, bbox_inches='tight')
plt.show()

# Part B: Full comet tail visualization
fig, ax = plt.subplots(figsize=(10, 10))

# Sun at origin
sun = Circle((0, 0), 0.08, color='yellow', ec='orange', linewidth=2, zorder=10)
ax.add_patch(sun)
ax.annotate('Sun\n태양', (0, 0), ha='center', va='center', fontsize=9, fontweight='bold', zorder=11)

# Comet orbit (parabolic approximation)
theta_orbit = np.linspace(-2.5, 2.5, 300)
r_orbit = 1.0 / (1 + np.cos(theta_orbit))  # Parabolic orbit, perihelion = 0.5 AU
x_orbit = r_orbit * np.cos(theta_orbit)
y_orbit = r_orbit * np.sin(theta_orbit)
mask = r_orbit < 4
ax.plot(x_orbit[mask], y_orbit[mask], 'gray', linewidth=1, alpha=0.5, linestyle='--')

# Show comet at several positions along orbit
positions = [-1.2, -0.8, -0.4, 0.0, 0.4, 0.8]
v_sw_nominal = 500  # km/s

for theta in positions:
    r = 1.0 / (1 + np.cos(theta))
    if r > 4:
        continue
    x_c = r * np.cos(theta)
    y_c = r * np.sin(theta)

    # Radial direction (anti-sunward)
    dx_rad = x_c / r
    dy_rad = y_c / r

    # Type I tail (ion): nearly anti-sunward with small aberration
    v_q = comet_transverse_velocity(r, 0)
    ab_angle = np.radians(aberration_angle(v_q, v_sw_nominal))

    # Orbital velocity direction (perpendicular to radius, prograde)
    dx_orb = -np.sin(theta)
    dy_orb = np.cos(theta)

    # Ion tail direction: anti-sunward + small aberration toward orbital trailing
    tail_len = 0.8
    dx_tail = dx_rad * np.cos(ab_angle) + dy_rad * np.sin(ab_angle)
    dy_tail = dy_rad * np.cos(ab_angle) - dx_rad * np.sin(ab_angle)

    # Draw ion tail (Type I)
    ax.annotate('', xy=(x_c + tail_len * dx_tail, y_c + tail_len * dy_tail),
                xytext=(x_c, y_c),
                arrowprops=dict(arrowstyle='->', color='blue', lw=2, alpha=0.7))

    # Draw dust tail (Type II) — more curved, lagging behind
    dust_angle = ab_angle * 5  # Dust tails have larger aberration
    dx_dust = dx_rad * np.cos(dust_angle) + dy_rad * np.sin(dust_angle)
    dy_dust = dy_rad * np.cos(dust_angle) - dx_rad * np.sin(dust_angle)
    ax.annotate('', xy=(x_c + 0.5 * dx_dust, y_c + 0.5 * dy_dust),
                xytext=(x_c, y_c),
                arrowprops=dict(arrowstyle='->', color='orange', lw=1.5, alpha=0.5))

    # Comet head
    ax.plot(x_c, y_c, 'ko', markersize=5, zorder=5)

# Legend
ax.plot([], [], 'b-', linewidth=2, label='Type I (ion tail / 이온 꼬리)')
ax.plot([], [], '-', color='orange', linewidth=1.5, label='Type II (dust tail / 먼지 꼬리)')
ax.legend(loc='lower left', fontsize=11)

ax.set_xlim(-2.5, 3.5)
ax.set_ylim(-3, 3)
ax.set_aspect('equal')
ax.set_xlabel('x [AU]')
ax.set_ylabel('y [AU]')
ax.set_title('Comet Tail Geometry: Ion Tails Always Point\nNearly Anti-Sunward / 이온 꼬리는 항상 거의 반태양 방향')
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig('part5_comet_visualization.png', dpi=150, bbox_inches='tight')
plt.show()

# Print aberration angles for typical cases
print("\nAberration angles at 1 AU (φ=0°) / 1 AU에서의 수차각:")
for v_sw in [400, 500, 1000, 2000]:
    vq = comet_transverse_velocity(1.0, 0)
    angle = aberration_angle(vq, v_sw)
    print(f"  v_sw = {v_sw:4d} km/s → aberration = {angle:.1f}°")

## Part 6: Solar Corpuscular Radiation Density Profile
## 파트 6: 태양 입자 복사 밀도 분포

Biermann estimated particle densities at various distances from the Sun. Assuming
a steady-state radial outflow with conservation of mass ($n \cdot v \cdot r^2 = \text{const}$),
we can trace the density from the corona to interplanetary space.

Biermann은 태양으로부터 다양한 거리에서 입자 밀도를 추정했습니다.
정상 상태 방사 유출과 질량 보존($n \cdot v \cdot r^2 = \text{const}$)을 가정하면,
코로나에서 행성간 공간까지의 밀도를 추적할 수 있습니다.

In [ ]:
# Biermann's density estimates at various distances
# Mass conservation for radial outflow: n * v * r² = const
# → n(r) = n_0 * (r_0/r)² * (v_0/v(r))

# Reference: inner corona
n_corona = 1e8     # cm⁻³ at ~1.1 R_sun
v_corona = 50e5    # 50 km/s (monochromatic corona outflow)
r_corona = 1.1     # R_sun

# Distance range: 1 R_sun to 5 AU
r_rsun = np.logspace(0, np.log10(5 * AU / R_sun), 500)
r_au = r_rsun * R_sun / AU

# Simple model: velocity increases from corona to asymptotic value
# v(r) = v_inf * (1 - R_sun/r)^alpha  (Parker-like, simplified)
v_inf = 500e5      # 500 km/s asymptotic
alpha_v = 0.5
v_profile = v_inf * (1 - 1.0/r_rsun)**alpha_v
v_profile[r_rsun < 1.01] = v_corona  # Inner corona

# Density from mass conservation
n_profile = n_corona * (r_corona / r_rsun)**2 * (v_corona / v_profile)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: Density profile
ax1 = axes[0]
ax1.loglog(r_au, n_profile, 'b-', linewidth=2.5, label='Model: n(r) from mass conservation')

# Biermann's data points
biermann_points = {
    'Inner corona\n(내부 코로나)': (1.1 * R_sun / AU, 1e8),
    'Storms at 1 AU\n(폭풍 시 1 AU)': (1.0, 1e5),
    'Normal 1 AU\n(정상 1 AU)': (1.0, 1e4),
    'Whipple limit\n(Whipple 상한)': (1.0, 1e3),
}
markers = ['r^', 'rs', 'ro', 'gv']
for (label, (r, n)), marker in zip(biermann_points.items(), markers):
    ax1.plot(r, n, marker, markersize=12, markeredgecolor='black',
             label=f'{label}: n={n:.0e}', zorder=5)

ax1.set_xlabel('Distance from Sun [AU] / 태양으로부터 거리')
ax1.set_ylabel('Particle density n [cm⁻³] / 입자 밀도')
ax1.set_title("Biermann's Density Estimates\nBiermann의 밀도 추정")
ax1.legend(fontsize=9, loc='upper right')
ax1.grid(True, alpha=0.3)
ax1.set_xlim(1e-3, 5)
ax1.set_ylim(1e0, 1e10)

# Right: Velocity profile
ax2 = axes[1]
ax2.semilogx(r_au, v_profile / 1e5, 'r-', linewidth=2.5, label='Model v(r)')

# Biermann's velocity estimates
v_points = {
    'Corona outflow\n(코로나 유출)': (1.1 * R_sun / AU, 50),
    'Quiet Sun\n(조용한 태양)': (1.0, 500),
    'Magnetic storm\n(자기 폭풍)': (1.0, 2000),
}
markers_v = ['b^', 'go', 'r*']
for (label, (r, v)), marker in zip(v_points.items(), markers_v):
    ax2.plot(r, v, marker, markersize=12, markeredgecolor='black',
             label=f'{label}: v={v} km/s', zorder=5)

ax2.set_xlabel('Distance from Sun [AU] / 태양으로부터 거리')
ax2.set_ylabel('Velocity [km/s] / 속도')
ax2.set_title("Velocity Profile\n속도 분포")
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)
ax2.set_xlim(1e-3, 5)
ax2.set_ylim(0, 2500)

plt.tight_layout()
plt.savefig('part6_density_profile.png', dpi=150, bbox_inches='tight')
plt.show()

# Mass flux calculation
print("Mass flux at 1 AU / 1 AU에서의 질량 플럭스:")
for n_1au, v_1au, label in [(1e4, 500e5, 'Normal'), (1e5, 2000e5, 'Storm')]:
    flux = n_1au * v_1au * m_p  # g/cm²/s
    total_flux = flux * 4 * np.pi * AU**2  # Total mass loss rate [g/s]
    mass_loss_per_year = total_flux * 3.15e7 / M_sun
    print(f"  {label}: n={n_1au:.0e}, v={v_1au/1e5:.0f} km/s")
    print(f"    Flux = {flux:.2e} g/cm²/s")
    print(f"    Total mass loss = {total_flux:.2e} g/s = {mass_loss_per_year:.2e} M_sun/yr")

## Part 7: Connection to Modern Solar Wind Measurements
## 파트 7: 현대 태양풍 관측과의 비교

Let's compare Biermann's 1951 estimates with modern in-situ solar wind measurements.
The solar wind was directly confirmed by Mariner 2 in 1962, and has been continuously
monitored since. Biermann's estimates were remarkably close to reality.

Biermann의 1951년 추정값을 현대 현장(in-situ) 태양풍 관측과 비교합니다.
태양풍은 1962년 Mariner 2에 의해 직접 확인되었으며, 이후 지속적으로 관측되고 있습니다.
Biermann의 추정값은 놀라울 정도로 실제에 가까웠습니다.

In [ ]:
# Comparison table: Biermann (1951) vs Modern measurements
# Modern values from typical solar wind conditions at 1 AU

comparison_data = {
    'Parameter': [
        'Proton density at 1 AU\n(양성자 밀도, 1 AU)',
        'Velocity (quiet Sun)\n(속도, 조용한 태양)',
        'Velocity (storms)\n(속도, 폭풍 시)',
        'Temperature\n(온도)',
        'Ion composition\n(이온 구성)',
        'Continuous flow?\n(연속 흐름?)',
    ],
    'Biermann (1951)': [
        '10³–10⁵ cm⁻³',
        '~500 km/s',
        '1000–2000 km/s',
        '~10⁴ K',
        'Ionized H (protons\n+ electrons)',
        'YES (first to claim!)',
    ],
    'Modern (in-situ)': [
        '3–10 cm⁻³',
        '300–450 km/s (slow)\n600–800 km/s (fast)',
        '500–2500 km/s (CME)',
        '~10⁵ K (protons)\n~1.5×10⁵ K (electrons)',
        '95% p⁺ + e⁻,\n~4% He²⁺, ~1% heavy',
        'YES (confirmed 1962\nby Mariner 2)',
    ],
    'Agreement': [
        'Overestimated by\n~10²–10⁴ ×',
        'Remarkably close!',
        'Good agreement',
        'Underestimated by\n~10× (protons)',
        'Correct (ionized)',
        'CORRECT!',
    ]
}

fig, ax = plt.subplots(figsize=(14, 8))
ax.axis('off')

# Create a comparison table
cell_colors = []
for i in range(len(comparison_data['Parameter'])):
    row_colors = ['#f0f0f0', '#e8f4fd', '#e8fde8', '#fff3e0']
    cell_colors.append(row_colors)

table = ax.table(
    cellText=[[comparison_data[col][i] for col in comparison_data.keys()]
              for i in range(len(comparison_data['Parameter']))],
    colLabels=list(comparison_data.keys()),
    cellLoc='center',
    loc='center',
    cellColours=cell_colors,
)

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.0, 2.5)

# Style header
for j in range(4):
    table[0, j].set_facecolor('#2c3e50')
    table[0, j].set_text_props(color='white', fontweight='bold')

ax.set_title("Biermann (1951) vs Modern Solar Wind Measurements\n"
             "Biermann (1951) vs 현대 태양풍 관측", fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig('part7_comparison_table.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary visualization: Biermann's density was too high, but the concept was right
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Density comparison
ax1 = axes[0]
categories = ['Biermann\n(quiet)\n(정상)', 'Biermann\n(storm)\n(폭풍)',
              'Modern\n(slow wind)\n(느린 풍)', 'Modern\n(fast wind)\n(빠른 풍)']
densities = [1e4, 1e5, 7, 3]
colors_d = ['#3498db', '#e74c3c', '#2ecc71', '#27ae60']
bars = ax1.bar(categories, densities, color=colors_d, edgecolor='black', alpha=0.8)
ax1.set_yscale('log')
ax1.set_ylabel('Proton density at 1 AU [cm⁻³]')
ax1.set_title('Density Estimates: Then vs Now\n밀도 추정: 과거 vs 현재')
ax1.set_ylim(1, 2e5)
for bar, val in zip(bars, densities):
    ax1.text(bar.get_x() + bar.get_width()/2, val * 1.5,
             f'{val:.0e}' if val > 100 else f'{val}',
             ha='center', fontweight='bold', fontsize=11)

# Right: Velocity comparison
ax2 = axes[1]
categories_v = ['Biermann\n(quiet)\n(정상)', 'Biermann\n(storm)\n(폭풍)',
                'Modern\n(slow)\n(느린)', 'Modern\n(fast)\n(빠른)']
velocities = [500, 1500, 400, 700]  # km/s
colors_v = ['#3498db', '#e74c3c', '#2ecc71', '#27ae60']
bars_v = ax2.bar(categories_v, velocities, color=colors_v, edgecolor='black', alpha=0.8)
ax2.set_ylabel('Velocity at 1 AU [km/s]')
ax2.set_title('Velocity Estimates: Then vs Now\n속도 추정: 과거 vs 현재')
for bar, val in zip(bars_v, velocities):
    ax2.text(bar.get_x() + bar.get_width()/2, val + 30,
             f'{val} km/s', ha='center', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.savefig('part7_comparison_bars.png', dpi=150, bbox_inches='tight')
plt.show()

print("=" * 60)
print("Summary / 요약")
print("=" * 60)
print("""
Biermann's 1951 predictions vs reality:
Biermann의 1951년 예측 vs 실제:

✓ CORRECT: The Sun continuously emits charged particles
  정확: 태양은 지속적으로 하전 입자를 방출한다

✓ CORRECT: Particles are ionized (protons + electrons)
  정확: 입자는 이온화되어 있다 (양성자 + 전자)

✓ CLOSE: Velocity estimates (500–2000 km/s vs 300–800 km/s)
  근접: 속도 추정 (500–2000 vs 300–800 km/s)

✗ HIGH: Density overestimated by ~10²–10⁴×
  과대추정: 밀도가 약 10²–10⁴배 높게 추정됨

✓ CORRECT: Electron-mediated momentum transfer to cometary ions
  정확: 전자가 매개하는 혜성 이온 운동량 전달

The density overestimate is understandable — Biermann had to
rely on indirect geomagnetic storm data. The key insight that
the Sun ALWAYS emits particles was revolutionary and correct.
밀도 과대추정은 이해할 만합니다 — Biermann은 간접적인 지자기 폭풍
데이터에 의존해야 했습니다. 태양이 항상 입자를 방출한다는 핵심
통찰은 혁명적이었고 정확했습니다.
""")

## Summary Table / 요약 테이블

| Part | Topic / 주제 | Key Result / 핵심 결과 |
|------|-------------|----------------------|
| 1 | Radiation pressure failure / 광압 실패 | μ_obs/μ_rad ≈ 10²–10³ for Type I tails |
| 2 | Coulomb vs gas-kinetic cross-sections / 충돌 단면적 비교 | σ_Coulomb(e⁻) ≫ σ_gas-kinetic by many orders of magnitude |
| 3 | Electron dominance / 전자 지배 | Electrons are the primary momentum carriers, not protons |
| 4 | Parameter space / 매개변수 공간 | Biermann's estimate matches observed CO⁺ acceleration |
| 5 | Tail geometry / 꼬리 기하학 | Aberration angle ~4–6° at 1 AU, consistent with observations |
| 6 | Density profile / 밀도 분포 | Mass conservation gives consistent picture from corona to 1 AU |
| 7 | Modern comparison / 현대 비교 | Velocity: close; Density: overestimated; Concept: correct |

**Next paper / 다음 논문**: Parker (1958) — "Dynamics of the Interplanetary Gas and Magnetic Fields"
→ Formalizes Biermann's corpuscular radiation as the theoretically predicted "solar wind"
→ Biermann의 corpuscular radiation을 이론적으로 예측된 "태양풍"으로 정립